# 113장 · 파일 백업과 세션 데이터 복원

백업 파일의 존재만으로는 복구 여부를 알 수 없습니다. 다른 위치에서 사본을 열고 필요한 레코드가 그대로 읽히는지 확인합니다.

임시 세션 사본을 별도 위치에 복원하고 저장된 요청을 다시 조회합니다. 원본 보존, 사본의 바이트 일치, 실제 업무 데이터의 복구 범위를 구분해 기록합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 별도 경로로 세션 사본 복원

복구 기록 예시　① 가상 세션 파일을 준비 ② 별도 디렉터리에 복구 ③ 복구 직후 원본과 바이트 대조 ④ 새 위치에서 마지막 요청 ID 읽기. 운영 상황에서는 복구 시점과 새 쓰기를 멈췄는지 먼저 확인해야 합니다. 아래 셀은 자신이 만든 임시 폴더 안에서만 파일을 만듭니다.

**원본과 별도 위치에 복원한 사본**　왼쪽 보관함에 원본이 남아 있는 동안 오른쪽 사본을 열어 봅니다. 복구 검증은 사본을 실제로 읽고 필요한 값을 대조하는 작업입니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter113/index.html#concept-clip)


### 셀 01 · 세션 사본의 복원과 재조회

읽기 확인은 복구 대상에서 수행합니다. 원본 내용으로 답을 출력하면 잘못된 복구 경로도 성공으로 보입니다.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json
from shutil import copyfile

with TemporaryDirectory() as temp:
    root = Path(temp)
    source = root / "source.json"
    source.write_text(json.dumps({"owner": "alice", "requests": ["r5", "r6"]}), encoding="utf-8")
    target_dir = root / "restored"
    target_dir.mkdir()
    target = target_dir / "session.json"
    copyfile(source, target)
    restored = json.loads(target.read_text(encoding="utf-8"))
    print(target.read_bytes() == source.read_bytes(), restored["requests"][-1])

True r6


True와 r6가 나옵니다. 첫 값은 현재 두 사본의 바이트 일치이고 둘째 값은 복구한 사본을 읽은 결과입니다. 셀이 끝나면 이 셀의 임시 디렉터리는 정리됩니다.

복원한 사본에서 요청 r6을 다시 읽은 것이 이번 검사의 범위입니다. 운영 복구 시간을 제시하려면 운영 백업을 대상으로 복원을 수행하고, 필요한 요청을 다시 조회할 때까지 걸린 시간을 기록해야 합니다. 로컬 사본의 성공과 운영 훈련 결과를 분리합니다.


### True와 r6이 확인한 복원 결과의 차이

True는 복사한 두 파일을 바이트 단위로 비교한 결과입니다. r6은 복구한 위치에서 JSON을 다시 읽고 마지막 요청 ID를 꺼낸 값입니다. 바이트가 같다는 검사와 복구 파일을 실제로 파싱해 원하는 기록을 찾는 검사를 함께 수행한 셈입니다.


## 복원한 레코드의 내용과 중복 검사

제공 가상 복구 기록에서는 alice가 유지한 요청이 r2~r6였고 다른 사용자의 같은 이름 세션은 0턴이었습니다. 복구 뒤에도 만료 시각은 그대로 두었습니다. 최근 r6을 다시 쓰려는 요청은 거부했고, 보존 범위를 벗어난 r0까지 영구 차단하지는 않았습니다. “파일이 돌아왔다”와 “오래된 중복 요청이 모두 차단된다”는 각각 다른 주장입니다.


### 과제 02 · 선택 연습 · 복원 범위 기록

제공 기록에서 원본이 그대로였는지와 운영 자료를 복구한 적이 있는지를 읽어 두 범위를 한 문장씩 구분하세요.

셀에서 읽은 마지막 값 r6와 임시 사본의 요청 r5·r6를 재현 기록에 적습니다. 보존 기간을 지켰는지와 같은 요청이 두 번 저장되지 않았는지는 별도 검사의 기록을 보고 채우세요.


In [2]:
record = {'original_unchanged_after_replay': True,
 'actual_restore_of_operational_data': False}
# 두 필드를 읽어 복구 대상과 결과를 표시하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>record = {&#x27;original_unchanged_after_replay&#x27;: True,
 &#x27;actual_restore_of_operational_data&#x27;: False}
print(&quot;가상 원본 보존&quot;, record[&quot;original_unchanged_after_replay&quot;])
print(&quot;운영 자료 실제 복구&quot;, record[&quot;actual_restore_of_operational_data&quot;])</code></pre>
<p>실행 결과</p><pre><code>가상 원본 보존 True
운영 자료 실제 복구 False</code></pre>
<p>가상 원본 보존은 True, 운영 자료 복구는 False입니다. 팀의 실환경 기록에는 복구 대상과 확인 시간을 다시 적습니다.</p>
</details>


## 돌아보기

114장에 인계할 기록은 복구 대상·복구한 위치·읽어 확인한 값·아직 모르는 범위입니다. 운영 장비의 재시작 시각과 실제 사용자 영향은 이 로컬 연습만으로 채우지 않습니다.
